# 🎨 Context-Aware Colorisation of Complex Scenes

**Internship Project** — Advanced AI-powered scene colorization using:
- **DeOldify** (base colorization backbone)
- **SegFormer-B2** (semantic segmentation)
- **YOLOv8** (object detection)
- **Scene-Aware Color Enhancement** (contextual reasoning engine)
- **Gradio GUI** (interactive web interface)

> Ensure you are using a **GPU runtime**: Runtime → Change runtime type → T4 GPU

---
## SECTION 1 — Install Dependencies

In [ ]:
# ============================================================
# SECTION 1 — Install Dependencies
# ============================================================
# Install all required packages for the pipeline.
# Run this cell first and wait for completion before proceeding.

import subprocess, sys

def run(cmd):
    """Run a shell command and stream output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] Command exited non-zero: {cmd}")
        print(result.stderr[-800:] if result.stderr else "")
    else:
        print(f"[OK] {cmd.split()[0]}")

print("Installing core dependencies...")
run("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
run("pip install -q transformers>=4.35.0 accelerate")
run("pip install -q ultralytics")
run("pip install -q opencv-python-headless Pillow numpy scipy scikit-image")
run("pip install -q gradio>=4.0.0")
run("pip install -q matplotlib tqdm requests")
run("pip install -q timm einops")
run("pip install -q fastai==2.7.13")   # DeOldify dependency

print("\nCloning DeOldify...")
run("git clone -q https://github.com/jantic/DeOldify.git DeOldify 2>/dev/null || echo 'Already cloned'")

print("\nDownloading DeOldify artistic colorizer weights...")
run("mkdir -p DeOldify/models")
run("""
if [ ! -f DeOldify/models/ColorizeArtistic_gen.pth ]; then
    wget -q -O DeOldify/models/ColorizeArtistic_gen.pth \
    'https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth'
    echo 'Weights downloaded'
else
    echo 'Weights already present'
fi
""")

print("\n✅ All dependencies installed.")

---
## SECTION 2 — Import Libraries

In [ ]:
# ============================================================
# SECTION 2 — Import Libraries
# ============================================================

import os, sys, time, warnings, traceback, shutil
from pathlib import Path
from typing import Optional, Tuple, Dict, List, Any

warnings.filterwarnings("ignore")

# --- Numeric / Image ---
import numpy as np
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import scipy.ndimage as ndimage
from skimage import color as skcolor

# --- PyTorch ---
import torch
import torch.nn as nn
import torchvision.transforms as T

# --- Hugging Face Transformers ---
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    pipeline as hf_pipeline,
)

# --- YOLO ---
from ultralytics import YOLO

# --- Visualization ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Gradio ---
import gradio as gr

# --- Add DeOldify to path ---
DEOLDIFY_PATH = Path("DeOldify")
if DEOLDIFY_PATH.exists() and str(DEOLDIFY_PATH) not in sys.path:
    sys.path.insert(0, str(DEOLDIFY_PATH))
    print(f"[OK] DeOldify path added: {DEOLDIFY_PATH.resolve()}")
else:
    print(f"[WARN] DeOldify folder not found at {DEOLDIFY_PATH.resolve()}")

# Detect GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n{'='*50}")
print(f"  Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  PyTorch: {torch.__version__}")
print(f"{'='*50}\n")
print("✅ Libraries imported successfully.")

---
## SECTION 3 — Configuration and Settings

In [ ]:
# ============================================================
# SECTION 3 — Configuration and Settings  (FIXED)
# ============================================================

from pathlib import Path
from typing import Optional, Tuple, Dict, List, Any

# ----------- Paths -----------
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
TEMP_DIR = Path("temp")
TEMP_DIR.mkdir(exist_ok=True)

# ----------- Model IDs -----------
SEGFORMER_MODEL_ID    = "nvidia/segformer-b2-finetuned-cityscapes-1024-1024"
SCENE_CLASSIFIER_ID   = "microsoft/resnet-50"
YOLO_MODEL_ID         = "yolov8n.pt"
DEOLDIFY_WEIGHTS_PATH = Path("DeOldify/models/ColorizeArtistic_gen.pth")

# ----------- Processing knobs -----------
MAX_IMAGE_SIZE         = 1024   # resize long-edge to this if larger
DEOLDIFY_RENDER_FACTOR = 35     # 21–45; higher = richer colour, more VRAM
BLEND_STRENGTH         = 0.55   # chroma blend strength [0-1] (lower = subtler)
SHADOW_DARKENING       = 0.60   # shadow brightness multiplier
REFLECTION_BLEND       = 0.35   # sky→water reflection intensity

# ----------- Scene colour palettes (BGR for OpenCV compatibility) -----------
# Values are (B, G, R) tuples targeting realistic scene colours.
# Only the a/b Lab channels are blended — L (luminance) is always preserved.
SCENE_COLOR_PALETTE: Dict[str, Dict] = {
    "urban": {
        "sky":        ( 210, 180, 140),   # warm blue sky
        "road":       (  88,  88,  88),   # neutral grey tarmac
        "sidewalk":   ( 145, 140, 135),   # slightly lighter grey
        "building":   ( 120, 115, 110),   # stone/concrete grey-beige
        "vegetation": (  30, 100,  38),   # urban greenery
        "terrain":    ( 100, 130,  80),   # grassy terrain
        "car":        ( 105, 100,  95),   # neutral grey cars
        "person":     ( 160, 130, 110),   # skin-tone neutral
    },
    "forest": {
        "sky":        ( 215, 185, 150),   # forest-filtered blue
        "vegetation": (  22, 108,  32),   # deep forest green
        "road":       (  78,  78,  72),   # shaded road
        "terrain":    (  55,  88,  44),   # forest floor
        "building":   ( 105,  95,  85),   # earthy building
        "car":        ( 100,  95,  90),
        "person":     ( 155, 125, 105),
    },
    "riverside": {
        "sky":        ( 200, 168, 140),   # open sky
        "terrain":    ( 145, 130,  90),   # river water (warm blue-grey)
        "vegetation": (  24, 105,  36),   # riverside vegetation
        "road":       (  80,  80,  76),
        "building":   ( 110, 100,  90),
        "car":        ( 100,  95,  90),
        "person":     ( 155, 125, 105),
    },
    "mountain": {
        "sky":        ( 210, 182, 148),   # high-altitude blue
        "terrain":    ( 135, 128, 118),   # rocky terrain grey
        "vegetation": (  26,  98,  34),   # mountain vegetation
        "road":       (  76,  76,  72),
        "building":   ( 112, 102,  92),
        "car":        ( 100,  95,  90),
        "person":     ( 150, 120, 100),
    },
    "park": {
        "sky":        ( 208, 178, 142),
        "vegetation": (  20, 110,  34),   # vibrant park grass/trees
        "terrain":    (  48,  95,  42),   # lush ground
        "road":       (  84,  84,  80),
        "sidewalk":   ( 146, 141, 138),
        "building":   ( 128, 118, 108),
        "car":        ( 102,  97,  92),
        "person":     ( 158, 128, 108),
    },
    "cityscape": {
        "sky":        ( 200, 172, 138),   # cityscape sky
        "building":   ( 130, 122, 115),   # urban facades
        "road":       (  85,  85,  90),   # wet-look city road
        "sidewalk":   ( 145, 140, 136),
        "vegetation": (  28,  96,  38),
        "terrain":    (  95, 125,  78),
        "car":        ( 115, 108, 102),
        "person":     ( 158, 128, 108),
    },
}
# 'default' falls back to urban
SCENE_COLOR_PALETTE["default"] = SCENE_COLOR_PALETTE["urban"]

print("✅ Configuration loaded.")
print(f"   Output directory : {OUTPUT_DIR.resolve()}")
print(f"   DeOldify weights : {DEOLDIFY_WEIGHTS_PATH.exists()}")
print(f"   Max image size   : {MAX_IMAGE_SIZE}px")
print(f"   Blend strength   : {BLEND_STRENGTH}")


---
## SECTION 4 — DeOldify Model Setup

In [ ]:
# ============================================================
# SECTION 4 — DeOldify Model Setup
# ============================================================
# DeOldify uses a U-Net generator trained with self-attention GAN.
# We load the Artistic colorizer which produces richer, more
# saturated results than the Stable variant.

class DeOldifyColorizer:
    """
    Wrapper around DeOldify's artistic colorizer.
    Falls back to a lightweight Lab-space colorization if
    DeOldify weights are unavailable (e.g. download failed).
    """

    def __init__(self, weights_path: Path, render_factor: int = DEOLDIFY_RENDER_FACTOR):
        self.render_factor = render_factor
        self.model = None
        self.available = False
        self._load(weights_path)

    # ----------------------------------------------------------
    def _load(self, weights_path: Path):
        """Attempt to load the DeOldify model."""
        try:
            # DeOldify requires fastai; import locally to avoid
            # polluting the global namespace on failure.
            from deoldify.visualize import get_image_colorizer
            if not weights_path.exists():
                raise FileNotFoundError(f"Weights not found: {weights_path}")
            self.model = get_image_colorizer(artistic=True)
            self.available = True
            print("[DeOldify] ✅ Artistic colorizer loaded.")
        except Exception as exc:
            print(f"[DeOldify] ⚠️  Could not load DeOldify: {exc}")
            print("[DeOldify] Falling back to Lab-space baseline colorizer.")
            self.available = False

    # ----------------------------------------------------------
    def colorize(self, pil_image: Image.Image) -> Image.Image:
        """
        Colorize a grayscale PIL image.
        Returns a color PIL image (RGB).
        """
        if self.available:
            return self._deoldify_colorize(pil_image)
        else:
            return self._lab_fallback(pil_image)

    # ----------------------------------------------------------
    def _deoldify_colorize(self, pil_image: Image.Image) -> Image.Image:
        """Use the loaded DeOldify model."""
        try:
            # Convert to grayscale then back to RGB (DeOldify expects RGB input)
            gray = pil_image.convert("L").convert("RGB")
            # Save temporarily — DeOldify's render_factor API needs a file path
            tmp_path = TEMP_DIR / "_deoldify_input.jpg"
            gray.save(str(tmp_path))
            result = self.model.get_transformed_image(
                path=str(tmp_path),
                render_factor=self.render_factor,
                compare=False,
                watermarked=False,
            )
            return result.convert("RGB")
        except Exception as exc:
            print(f"[DeOldify] Runtime error: {exc} — using fallback.")
            return self._lab_fallback(pil_image)

    # ----------------------------------------------------------
    def _lab_fallback(self, pil_image: Image.Image) -> Image.Image:
        """
        Baseline colorizer using Lab colour space.
        Produces a warm sepia-toned base that the context engine
        will enhance in later stages.
        """
        gray_np = np.array(pil_image.convert("L"))  # (H, W) uint8
        # Build a 3-channel L image
        L = gray_np.astype(np.float32)
        # Create plausible a/b channels with a neutral (slightly warm) bias
        a_chan = np.full_like(L, 3.0)   # slight red/green shift
        b_chan = np.full_like(L, 8.0)   # slight blue/yellow shift (warmer)
        # Rescale L to [0, 100]
        L_scaled = L / 255.0 * 100.0
        lab = np.stack([L_scaled, a_chan, b_chan], axis=-1).astype(np.float32)
        rgb = skcolor.lab2rgb(lab)      # [0, 1] float
        rgb_uint8 = (rgb * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(rgb_uint8, "RGB")


# Instantiate
print("Loading DeOldify...")
deoldify_model = DeOldifyColorizer(DEOLDIFY_WEIGHTS_PATH, DEOLDIFY_RENDER_FACTOR)
print(f"DeOldify available: {deoldify_model.available}")

---
## SECTION 5 — Semantic Segmentation Module

In [ ]:
# ============================================================
# SECTION 5 — Semantic Segmentation Module  (FIXED)
# ============================================================
# Uses NVIDIA SegFormer-B2 fine-tuned on Cityscapes (19 classes).
# Correct Cityscapes colour palette is used for visualisation.

class SemanticSegmentationModule:
    """
    Wraps SegFormer-B2 for per-pixel scene understanding.
    """

    # Official Cityscapes 19-class colour palette (RGB)
    CITYSCAPES_COLORS = np.array([
        [128,  64, 128],   # 0  road
        [244,  35, 232],   # 1  sidewalk
        [ 70,  70,  70],   # 2  building
        [102, 102, 156],   # 3  wall
        [190, 153, 153],   # 4  fence
        [153, 153, 153],   # 5  pole
        [250, 170,  30],   # 6  traffic light
        [220, 220,   0],   # 7  traffic sign
        [107, 142,  35],   # 8  vegetation
        [152, 251, 152],   # 9  terrain
        [ 70, 130, 180],   # 10 sky
        [220,  20,  60],   # 11 person
        [255,   0,   0],   # 12 rider
        [  0,   0, 142],   # 13 car
        [  0,   0,  70],   # 14 truck
        [  0,  60, 100],   # 15 bus
        [  0,  80, 100],   # 16 train
        [  0,   0, 230],   # 17 motorcycle
        [119,  11,  32],   # 18 bicycle
    ], dtype=np.uint8)

    def __init__(self, model_id: str = SEGFORMER_MODEL_ID):
        self.model_id = model_id
        self.processor = None
        self.model = None
        self.id2label: Dict[int, str] = {}
        self._load()

    def _load(self):
        """Download and cache SegFormer-B2."""
        print(f"[SegFormer] Loading {self.model_id} ...")
        self.processor = SegformerImageProcessor.from_pretrained(self.model_id)
        self.model = SegformerForSemanticSegmentation.from_pretrained(self.model_id)
        self.model.eval().to(DEVICE)
        self.id2label = self.model.config.id2label
        print(f"[SegFormer] ✅ Loaded. Classes: {len(self.id2label)}")
        print(f"[SegFormer] Label map: {dict(list(self.id2label.items())[:5])} ...")

    @torch.no_grad()
    def segment(self, pil_image: Image.Image) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run segmentation on a PIL image.

        Returns:
            seg_map : (H, W) int32  — class indices 0-18
            seg_vis : (H, W, 3) uint8 — colour-coded visualization
        """
        orig_w, orig_h = pil_image.size
        inputs = self.processor(images=pil_image.convert("RGB"), return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = self.model(**inputs)

        # Upsample logits to original image size
        logits = outputs.logits  # (1, num_classes, H/4, W/4)
        upsampled = torch.nn.functional.interpolate(
            logits,
            size=(orig_h, orig_w),
            mode="bilinear",
            align_corners=False,
        )
        # Use int32 to safely cover class indices
        seg_map = upsampled.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int32)
        seg_vis = self._colorize_segmap(seg_map)
        return seg_map, seg_vis

    def _colorize_segmap(self, seg_map: np.ndarray) -> np.ndarray:
        """Assign the official Cityscapes colour to each class."""
        H, W = seg_map.shape
        vis = np.zeros((H, W, 3), dtype=np.uint8)
        num_colors = len(self.CITYSCAPES_COLORS)
        for class_id in range(num_colors):
            mask = (seg_map == class_id)
            if mask.any():
                vis[mask] = self.CITYSCAPES_COLORS[class_id]
        return vis

    def get_class_mask(self, seg_map: np.ndarray, class_name: str) -> np.ndarray:
        """
        Return a boolean mask for a given class name substring.
        e.g. 'vegetation' matches id2label entry 'vegetation'
        """
        mask = np.zeros(seg_map.shape, dtype=bool)
        for idx, label in self.id2label.items():
            if class_name.lower() in label.lower():
                mask |= (seg_map == idx)
        return mask

    def get_detected_classes(self, seg_map: np.ndarray) -> Dict[str, float]:
        """
        Return {class_name: area_percent} for classes covering >= 1% of image.
        Uses the model's own id2label (not hard-coded strings).
        """
        total = float(seg_map.size)
        detected = {}
        for idx, label in self.id2label.items():
            frac = float((seg_map == idx).sum()) / total
            if frac >= 0.01:
                # Clean up label: strip numeric prefix if present (e.g. "8 vegetation")
                clean = label.split(" ")[-1] if " " in label else label
                detected[clean] = round(frac * 100, 1)
        return detected


print("Loading SegFormer-B2...")
seg_module = SemanticSegmentationModule()
print(f"Segmentation classes: {list(seg_module.id2label.values())}")


---
## SECTION 6 — Scene Classification Module

In [ ]:
# ============================================================
# SECTION 6 — Scene Classification Module
# ============================================================
# Two-stage scene classification:
#   1. Heuristic rule engine using SegFormer class area fractions
#      (deterministic, interpretable, no extra model needed)
#   2. Optional HuggingFace image-classification as secondary signal

class SceneClassifier:
    """
    Classifies scenes into: urban, forest, riverside, mountain, park, cityscape.
    Combines segmentation statistics with a lightweight HF classifier.
    """

    SCENE_TYPES = ["urban", "forest", "riverside", "mountain", "park", "cityscape", "default"]

    # Keywords from HF image-classification labels mapped to our scene types
    HF_LABEL_MAP = {
        "street":      "urban",
        "road":        "urban",
        "city":        "cityscape",
        "building":    "cityscape",
        "skyscraper":  "cityscape",
        "forest":      "forest",
        "tree":        "forest",
        "river":       "riverside",
        "lake":        "riverside",
        "water":       "riverside",
        "mountain":    "mountain",
        "valley":      "mountain",
        "park":        "park",
        "garden":      "park",
        "grass":       "park",
    }

    def __init__(self):
        self.hf_clf = None
        self._load_hf()

    def _load_hf(self):
        """Load a lightweight HF image classifier for secondary signal."""
        try:
            self.hf_clf = hf_pipeline(
                "image-classification",
                model="microsoft/resnet-50",
                device=0 if DEVICE.type == "cuda" else -1,
            )
            print("[SceneClassifier] ✅ ResNet-50 classifier loaded.")
        except Exception as exc:
            print(f"[SceneClassifier] ⚠️  HF classifier unavailable: {exc}")
            self.hf_clf = None

    # ----------------------------------------------------------
    def classify(
        self,
        pil_image: Image.Image,
        class_areas: Dict[str, float],
    ) -> str:
        """
        Determine scene type.

        Args:
            pil_image   : PIL image for HF classifier
            class_areas : {class_name: area_percent} from segmentation

        Returns:
            scene_type string
        """
        # --- Stage 1: Heuristic rules ---
        heuristic = self._heuristic_classify(class_areas)

        # --- Stage 2: HF classifier (soft vote) ---
        hf_vote = self._hf_classify(pil_image)

        # Merge: if both agree, done; otherwise prefer heuristic
        if heuristic == hf_vote:
            return heuristic
        elif heuristic != "default":
            return heuristic
        elif hf_vote:
            return hf_vote
        return "urban"  # ultimate fallback

    # ----------------------------------------------------------
    def _heuristic_classify(self, areas: Dict[str, float]) -> str:
        """
        Rule-based classification from segmentation area statistics.
        """
        veg   = areas.get("vegetation", 0)
        water = areas.get("water", 0) + areas.get("terrain", 0) * 0.3
        sky   = areas.get("sky", 0)
        road  = areas.get("road", 0) + areas.get("sidewalk", 0)
        bldg  = areas.get("building", 0)
        car   = areas.get("car", 0) + areas.get("bus", 0) + areas.get("truck", 0)

        # Forest: dominant vegetation, low road/building
        if veg > 45 and road < 15 and bldg < 10:
            return "forest"
        # Riverside: significant water
        if water > 20:
            return "riverside"
        # Mountain: sky + terrain dominant, low urban
        if sky > 30 and bldg < 15 and road < 15:
            return "mountain"
        # Park: veg + terrain, moderate sky, low road
        if veg > 25 and road < 25 and bldg < 20:
            return "park"
        # Cityscape: dense buildings + sky
        if bldg > 30 and sky > 10:
            return "cityscape"
        # Urban: roads + buildings + cars
        if road > 20 or (bldg > 15 and car > 5):
            return "urban"
        return "default"

    # ----------------------------------------------------------
    def _hf_classify(self, pil_image: Image.Image) -> Optional[str]:
        """Use HF classifier and map top-1 label to a scene type."""
        if self.hf_clf is None:
            return None
        try:
            results = self.hf_clf(pil_image.convert("RGB"), top_k=5)
            for r in results:
                label_lower = r["label"].lower()
                for keyword, scene in self.HF_LABEL_MAP.items():
                    if keyword in label_lower:
                        return scene
        except Exception:
            pass
        return None


print("Loading scene classifier...")
scene_classifier = SceneClassifier()
print("✅ Scene classification module ready.")

---
## SECTION 7 — Context-Aware Colorization Engine

In [ ]:
# ============================================================
# SECTION 7 — Context-Aware Colorization Engine  (FIXED)
# ============================================================

class ContextAwareColorizer:
    """
    Applies scene-specific colour palette to each semantic region.
    Blends only the chrominance (a/b) channels in Lab space so
    texture and luminance are perfectly preserved.
    """

    # Maps normalised label substrings → palette dictionary key
    # Normalised = lower-case, strip leading digits/spaces
    LABEL_TO_PALETTE_KEY = {
        "sky":        "sky",
        "vegetation": "vegetation",
        "terrain":    "terrain",
        "road":       "road",
        "sidewalk":   "sidewalk",
        "building":   "building",
        "wall":       "building",
        "fence":      "building",
        "car":        "car",
        "truck":      "car",
        "bus":        "car",
        "person":     "person",
        "rider":      "person",
    }

    @staticmethod
    def _normalise_label(label: str) -> str:
        """Strip leading 'N ' prefixes like '8 vegetation' → 'vegetation'."""
        parts = label.strip().split()
        # If first token is a digit, skip it
        if parts and parts[0].isdigit():
            return " ".join(parts[1:]).lower()
        return label.lower()

    def enhance(
        self,
        deoldify_rgb: np.ndarray,   # (H, W, 3) uint8
        gray_image: np.ndarray,     # (H, W) uint8 — original luminance
        seg_map: np.ndarray,        # (H, W) int32 — class indices
        scene_type: str,
        id2label: Dict[int, str],
    ) -> np.ndarray:
        """
        Main entry point.
        Returns enhanced (H, W, 3) uint8 RGB image.
        """
        palette = SCENE_COLOR_PALETTE.get(scene_type, SCENE_COLOR_PALETTE["default"])

        # Work in Lab space for perceptual colour blending
        # OpenCV Lab: L in [0,255], a/b in [0,255] centred at 128
        result_lab = cv2.cvtColor(deoldify_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

        for class_idx, label_raw in id2label.items():
            norm_label = self._normalise_label(label_raw)

            # Find matching palette key
            palette_key = None
            for seg_label, pk in self.LABEL_TO_PALETTE_KEY.items():
                if seg_label in norm_label:
                    palette_key = pk
                    break

            if palette_key is None or palette_key not in palette:
                continue

            mask = (seg_map == class_idx)
            if mask.sum() < 100:   # skip tiny regions
                continue

            # Build smooth edge mask
            mask_f = mask.astype(np.float32)
            mask_f = cv2.GaussianBlur(mask_f, (21, 21), 7)

            # Convert target BGR palette colour → Lab
            target_bgr = np.array(palette[palette_key], dtype=np.uint8).reshape(1, 1, 3)
            target_lab = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
            target_a = float(target_lab[0, 0, 1])
            target_b = float(target_lab[0, 0, 2])

            # Blend a/b (chroma) channels only — never touch L (luminance)
            blend = BLEND_STRENGTH
            result_lab[:, :, 1] = (
                result_lab[:, :, 1] * (1.0 - blend * mask_f)
                + target_a * blend * mask_f
            )
            result_lab[:, :, 2] = (
                result_lab[:, :, 2] * (1.0 - blend * mask_f)
                + target_b * blend * mask_f
            )

        # Convert back to RGB
        result_lab_u8 = np.clip(result_lab, 0, 255).astype(np.uint8)
        enhanced_rgb = cv2.cvtColor(result_lab_u8, cv2.COLOR_LAB2RGB)

        # Restore original luminance exactly (texture / edge preservation)
        enhanced_rgb = self._restore_luminance(enhanced_rgb, gray_image)
        return enhanced_rgb

    @staticmethod
    def _restore_luminance(
        colored: np.ndarray,   # (H, W, 3) uint8 RGB
        gray: np.ndarray,      # (H, W) uint8
    ) -> np.ndarray:
        """
        Replace L channel with original grayscale luminance.
        This guarantees texture and edge fidelity regardless of colour changes.
        """
        lab = cv2.cvtColor(colored, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = gray
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


# ── YOLO Object Detector ──────────────────────────────────────────
class ObjectDetector:
    """YOLOv8 nano for fast object detection statistics."""

    def __init__(self, model_id: str = YOLO_MODEL_ID):
        self.model = None
        self._load(model_id)

    def _load(self, model_id: str):
        try:
            self.model = YOLO(model_id)
            print(f"[YOLO] ✅ {model_id} loaded.")
        except Exception as exc:
            print(f"[YOLO] ⚠️  Load failed: {exc}")

    def detect(self, pil_image: Image.Image) -> Tuple[int, List[str]]:
        """Returns (count, [class_names]) of detected objects."""
        if self.model is None:
            return 0, []
        try:
            img_rgb = np.array(pil_image.convert("RGB"))
            results = self.model(img_rgb, verbose=False, conf=0.20)
            names = []
            for r in results:
                for cls_id in r.boxes.cls.cpu().numpy().astype(int):
                    names.append(self.model.names[int(cls_id)])
            return len(names), names
        except Exception as exc:
            print(f"[YOLO] Detection error: {exc}")
            return 0, []


print("Loading context-aware colorizer & YOLO...")
ctx_colorizer = ContextAwareColorizer()
object_detector = ObjectDetector()
print("✅ Colorization engine ready.")


---
## SECTION 8 — Reflection and Shadow Processing

In [ ]:
# ============================================================
# SECTION 8 — Reflection and Shadow Processing  (FIXED)
# ============================================================

class ReflectionHandler:
    """Sky-to-water colour reflection and shadow darkening."""

    @staticmethod
    def _normalise_label(label: str) -> str:
        """Strip leading digit prefix e.g. '10 sky' → 'sky'."""
        parts = label.strip().split()
        if parts and parts[0].isdigit():
            return " ".join(parts[1:]).lower()
        return label.lower()

    def _class_mask(
        self,
        seg_map: np.ndarray,
        id2label: Dict[int, str],
        class_name: str,
    ) -> np.ndarray:
        """Return boolean mask for all classes containing class_name."""
        mask = np.zeros(seg_map.shape, dtype=bool)
        for idx, label in id2label.items():
            norm = self._normalise_label(label)
            if class_name.lower() in norm:
                mask |= (seg_map == idx)
        return mask

    def apply_sky_water_reflection(
        self,
        rgb_image: np.ndarray,  # (H, W, 3) uint8
        seg_map: np.ndarray,
        id2label: Dict[int, str],
    ) -> np.ndarray:
        """
        Sample the dominant sky colour and blend it into terrain/water regions
        to simulate realistic reflections.
        """
        result = rgb_image.copy()

        sky_mask     = self._class_mask(seg_map, id2label, "sky")
        terrain_mask = self._class_mask(seg_map, id2label, "terrain")

        if sky_mask.sum() < 200 or terrain_mask.sum() < 200:
            return result  # not enough sky or water — skip

        # Robust sky colour: median of sky pixels
        sky_pixels = result[sky_mask]  # (N, 3) RGB
        sky_color_rgb = np.median(sky_pixels, axis=0).astype(np.uint8)  # (3,)

        # Build smooth terrain mask
        terrain_mask_f = cv2.GaussianBlur(
            terrain_mask.astype(np.float32), (25, 25), 9
        )

        # Work in HSV: blend hue and saturation of sky into terrain
        result_hsv = cv2.cvtColor(result, cv2.COLOR_RGB2HSV).astype(np.float32)

        sky_bgr    = sky_color_rgb[[2, 1, 0]].reshape(1, 1, 3).astype(np.uint8)
        sky_hsv    = cv2.cvtColor(sky_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
        sky_h      = float(sky_hsv[0, 0, 0])
        sky_s      = float(sky_hsv[0, 0, 1])

        alpha = REFLECTION_BLEND
        result_hsv[:, :, 0] = (
            result_hsv[:, :, 0] * (1.0 - alpha * terrain_mask_f)
            + sky_h * alpha * terrain_mask_f
        )
        result_hsv[:, :, 1] = (
            result_hsv[:, :, 1] * (1.0 - alpha * 0.6 * terrain_mask_f)
            + sky_s * alpha * 0.6 * terrain_mask_f
        )

        reflected = cv2.cvtColor(
            np.clip(result_hsv, 0, 255).astype(np.uint8),
            cv2.COLOR_HSV2RGB
        )
        mask3 = terrain_mask_f[:, :, np.newaxis]
        result = (result * (1.0 - mask3) + reflected * mask3).clip(0, 255).astype(np.uint8)
        return result

    def apply_shadow_darkening(
        self,
        rgb_image: np.ndarray,   # (H, W, 3) uint8
        gray_image: np.ndarray,  # (H, W) uint8
    ) -> np.ndarray:
        """
        Detect relative shadow regions (dark relative to local neighbourhood)
        and apply darkening without colour-casting.
        """
        result = rgb_image.astype(np.float32)

        gray_f  = gray_image.astype(np.float32)
        blurred = cv2.GaussianBlur(gray_f, (61, 61), 20)

        # Shadow = pixel is significantly darker than its neighbourhood
        shadow_map = np.clip(
            (blurred - gray_f) / (blurred + 1e-6), 0.0, 1.0
        )
        # Only keep strong shadows (threshold 0.18)
        shadow_map = np.where(shadow_map > 0.18, shadow_map, 0.0)
        shadow_map = cv2.GaussianBlur(shadow_map, (11, 11), 4)

        shadow3 = shadow_map[:, :, np.newaxis]
        result  = result * (1.0 - shadow3 * (1.0 - SHADOW_DARKENING))
        return np.clip(result, 0, 255).astype(np.uint8)


reflection_handler = ReflectionHandler()
print("✅ Reflection and shadow module ready.")


---
## SECTION 9 — Image Processing Pipeline

In [ ]:
# ============================================================
# SECTION 9 — Image Processing Pipeline
# ============================================================
# Orchestrates all modules into a single end-to-end pipeline.

def preprocess_image(pil_image: Image.Image) -> Tuple[Image.Image, np.ndarray, int, int]:
    """
    Resize to MAX_IMAGE_SIZE (long edge), convert to RGB.
    Returns: (pil_image_resized, gray_array, orig_w, orig_h)
    """
    orig_w, orig_h = pil_image.size
    pil_image = pil_image.convert("RGB")
    long_edge = max(orig_w, orig_h)
    if long_edge > MAX_IMAGE_SIZE:
        scale = MAX_IMAGE_SIZE / long_edge
        new_w, new_h = int(orig_w * scale), int(orig_h * scale)
        pil_image = pil_image.resize((new_w, new_h), Image.LANCZOS)
    gray = np.array(pil_image.convert("L"))  # (H, W) uint8
    return pil_image, gray, orig_w, orig_h


def run_pipeline(pil_image: Image.Image) -> Dict[str, Any]:
    """
    Full colorization pipeline.

    Args:
        pil_image : PIL Image (any mode/size)

    Returns dict with keys:
        gray_pil         : PIL grayscale image
        seg_vis_pil      : PIL segmentation visualization
        deoldify_pil     : PIL DeOldify colorized
        enhanced_pil     : PIL context-aware enhanced
        scene_type       : str
        class_areas      : dict
        object_count     : int
        object_names     : list
        processing_time  : float
        num_seg_classes  : int
    """
    t0 = time.time()

    # ── 1. Preprocess ───────────────────────────────────────
    image_rgb, gray_np, orig_w, orig_h = preprocess_image(pil_image)
    h, w = gray_np.shape

    # ── 2. Semantic Segmentation ────────────────────────────
    seg_map, seg_vis = seg_module.segment(image_rgb)
    class_areas = seg_module.get_detected_classes(seg_map)

    # ── 3. Scene Classification ─────────────────────────────
    scene_type = scene_classifier.classify(image_rgb, class_areas)

    # ── 4. Object Detection (statistics) ────────────────────
    obj_count, obj_names = object_detector.detect(image_rgb)

    # ── 5. DeOldify Colorization ─────────────────────────────
    deoldify_pil = deoldify_model.colorize(image_rgb)
    deoldify_rgb = np.array(deoldify_pil.convert("RGB"))

    # Ensure shape matches (DeOldify may resize internally)
    if deoldify_rgb.shape[:2] != (h, w):
        deoldify_rgb = cv2.resize(deoldify_rgb, (w, h), interpolation=cv2.INTER_LANCZOS4)

    # ── 6. Context-Aware Enhancement ────────────────────────
    enhanced_rgb = ctx_colorizer.enhance(
        deoldify_rgb,
        gray_np,
        seg_map,
        scene_type,
        seg_module.id2label,
    )

    # ── 7. Reflection + Shadow Post-Processing ───────────────
    enhanced_rgb = reflection_handler.apply_sky_water_reflection(
        enhanced_rgb, seg_map, seg_module.id2label
    )
    enhanced_rgb = reflection_handler.apply_shadow_darkening(
        enhanced_rgb, gray_np
    )

    # ── 8. Final sharpening (preserve edges) ────────────────
    enhanced_pil = Image.fromarray(enhanced_rgb)
    # Unsharp mask for edge crisp-ness
    enhanced_pil = enhanced_pil.filter(
        ImageFilter.UnsharpMask(radius=1.5, percent=110, threshold=3)
    )

    processing_time = round(time.time() - t0, 2)

    return {
        "gray_pil":        image_rgb.convert("L"),
        "seg_vis_pil":     Image.fromarray(seg_vis, "RGB"),
        "deoldify_pil":    deoldify_pil,
        "enhanced_pil":    enhanced_pil,
        "scene_type":      scene_type,
        "class_areas":     class_areas,
        "object_count":    obj_count,
        "object_names":    obj_names,
        "processing_time": processing_time,
        "num_seg_classes": len(class_areas),
    }


def validate_image(file_obj) -> Tuple[Optional[Image.Image], str]:
    """
    Validate uploaded file.
    Returns (PIL Image, error_msg).
    error_msg is empty string on success.
    """
    if file_obj is None:
        return None, "❌ No image uploaded. Please upload an image."
    try:
        img = Image.open(file_obj)
        img.verify()  # check for corrupt headers
        img = Image.open(file_obj)  # reopen after verify
        if img.size[0] < 32 or img.size[1] < 32:
            return None, "❌ Image too small (minimum 32×32 pixels)."
        if img.mode not in ("L", "RGB", "RGBA", "P", "1"):
            return None, f"❌ Unsupported image mode: {img.mode}"
        return img, ""
    except Exception as exc:
        return None, f"❌ Invalid or corrupted image: {exc}"


print("✅ Pipeline functions defined.")

---
## SECTION 10 — Gradio GUI

In [ ]:
# ============================================================
# SECTION 10 — Gradio GUI
# ============================================================

# ----------- Inference Wrapper -----------
def gradio_process(image_input):
    """
    Gradio callback.
    Input:  PIL Image from Gradio Image component
    Output: (gray, seg_vis, deoldify, enhanced, stats_text)
    """
    # Gradio Image component returns a PIL image directly
    if image_input is None:
        empty = Image.new("RGB", (400, 300), (30, 30, 30))
        msg = "⚠️ Please upload an image first."
        return empty, empty, empty, empty, msg

    try:
        pil_img = image_input if isinstance(image_input, Image.Image) \
                  else Image.fromarray(image_input)
        results = run_pipeline(pil_img)

        # Build statistics text
        top_classes = sorted(
            results["class_areas"].items(),
            key=lambda x: x[1],
            reverse=True,
        )[:6]
        class_lines = "\n".join([
            f"  • {k:15s} {v:5.1f}%"
            for k, v in top_classes
        ])

        obj_summary = ", ".join(
            sorted(set(results["object_names"]))
        ) or "None detected"

        stats_text = (
            f"🏞️  Scene Type       : {results['scene_type'].upper()}\n"
            f"⏱️  Processing Time  : {results['processing_time']} s\n"
            f"📦  Objects Detected : {results['object_count']}\n"
            f"🗂️  Object Classes   : {obj_summary}\n"
            f"🎨  Semantic Classes : {results['num_seg_classes']}\n\n"
            f"Top Semantic Segments:\n{class_lines}"
        )

        # Auto-save enhanced output
        save_outputs(results)

        return (
            results["gray_pil"],
            results["seg_vis_pil"],
            results["deoldify_pil"],
            results["enhanced_pil"],
            stats_text,
        )

    except Exception as exc:
        tb = traceback.format_exc()
        print(f"[Pipeline Error]\n{tb}")
        empty = Image.new("RGB", (400, 300), (40, 20, 20))
        return empty, empty, empty, empty, f"❌ Processing failed:\n{exc}\n\nCheck console for details."


# ----------- GUI Layout -----------
TITLE = "🎨 Context-Aware Colorisation of Complex Scenes"
DESCRIPTION = """
Upload any **grayscale or colour-reduced** photograph of a complex scene
(cityscape, forest, riverside, mountain, park, street) and the system will:

1. **Classify the scene** (urban / forest / mountain / riverside / park / cityscape)
2. **Segment semantic regions** using SegFormer-B2 (19 Cityscapes classes)
3. **Apply DeOldify** base colorization (self-attention GAN)
4. **Enhance with context-aware colours** — sky, water, vegetation, roads, buildings
5. **Simulate reflections** (sky → water) and **shadow darkening**
"""

with gr.Blocks(
    title=TITLE,
    theme=gr.themes.Soft(primary_hue="indigo"),
    css=".output-image img { border-radius: 8px; } "
        "#stats-box { font-family: monospace; font-size: 0.9em; }"
) as demo:

    gr.Markdown(f"# {TITLE}")
    gr.Markdown(DESCRIPTION)

    with gr.Row():
        # --- Left column: Input ---
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Input")
            input_image = gr.Image(
                label="Upload Image",
                type="pil",
                height=320,
                elem_classes=["output-image"],
            )
            run_btn = gr.Button(
                "🚀 Colorize",
                variant="primary",
                size="lg",
            )
            gr.Markdown("*Accepts JPEG, PNG, WEBP, BMP, TIFF*")

        # --- Right column: Statistics ---
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Scene Statistics")
            stats_box = gr.Textbox(
                label="Analysis Results",
                lines=14,
                interactive=False,
                placeholder="Run colorization to see statistics…",
                elem_id="stats-box",
            )

    gr.Markdown("---")
    gr.Markdown("### 🖼️ Pipeline Outputs")

    with gr.Row():
        gray_out = gr.Image(
            label="Grayscale Input",
            elem_classes=["output-image"],
            height=256,
        )
        seg_out = gr.Image(
            label="Semantic Segmentation Map",
            elem_classes=["output-image"],
            height=256,
        )
        deoldify_out = gr.Image(
            label="DeOldify Colorized",
            elem_classes=["output-image"],
            height=256,
        )
        enhanced_out = gr.Image(
            label="✨ Context-Aware Enhanced",
            elem_classes=["output-image"],
            height=256,
        )

    gr.Markdown("---")
    gr.Markdown("### ⬇️ Download")

    with gr.Row():
        download_file = gr.File(
            label="Download Enhanced Image",
            visible=False,
        )
        download_btn = gr.Button("📥 Prepare Download", variant="secondary")

    # --- Wire up events ---
    run_btn.click(
        fn=gradio_process,
        inputs=[input_image],
        outputs=[gray_out, seg_out, deoldify_out, enhanced_out, stats_box],
    )

    def prepare_download(enhanced_img):
        if enhanced_img is None:
            return gr.update(visible=False)
        out_path = str(OUTPUT_DIR / "latest_enhanced.png")
        if isinstance(enhanced_img, np.ndarray):
            Image.fromarray(enhanced_img).save(out_path)
        else:
            enhanced_img.save(out_path)
        return gr.update(value=out_path, visible=True)

    download_btn.click(
        fn=prepare_download,
        inputs=[enhanced_out],
        outputs=[download_file],
    )

    gr.Examples(
        examples=[
            # Add local image paths here if available
        ],
        inputs=input_image,
        label="Example Images (add paths above to enable)",
    )

print("✅ Gradio interface built.")

---
## SECTION 11 — Output Saving and Download System

In [ ]:
# ============================================================
# SECTION 11 — Output Saving and Download System
# ============================================================

import json

def save_outputs(results: Dict[str, Any]) -> Dict[str, str]:
    """
    Save all pipeline outputs to OUTPUT_DIR with a timestamp prefix.

    Returns dict of {label: file_path}.
    """
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    saved = {}

    # Images
    image_map = {
        "gray":      results["gray_pil"],
        "segmap":    results["seg_vis_pil"],
        "deoldify":  results["deoldify_pil"],
        "enhanced":  results["enhanced_pil"],
    }
    for label, img in image_map.items():
        if img is not None:
            path = OUTPUT_DIR / f"{timestamp}_{label}.png"
            img.save(str(path))
            saved[label] = str(path)

    # Always keep latest_enhanced.png for the download button
    if results.get("enhanced_pil") is not None:
        results["enhanced_pil"].save(str(OUTPUT_DIR / "latest_enhanced.png"))

    # Metadata JSON
    meta = {
        "timestamp":       timestamp,
        "scene_type":      results["scene_type"],
        "object_count":    results["object_count"],
        "object_names":    results["object_names"],
        "num_seg_classes": results["num_seg_classes"],
        "class_areas":     results["class_areas"],
        "processing_time": results["processing_time"],
        "saved_files":     saved,
    }
    meta_path = OUTPUT_DIR / f"{timestamp}_metadata.json"
    with open(str(meta_path), "w") as f:
        json.dump(meta, f, indent=2)
    saved["metadata"] = str(meta_path)

    print(f"[Save] ✅ Outputs saved to {OUTPUT_DIR}/ ({len(saved)} files)")
    return saved


def list_saved_outputs() -> List[str]:
    """List all files in the output directory."""
    files = sorted(OUTPUT_DIR.glob("*"))
    return [str(f) for f in files]


def download_from_colab(file_path: str):
    """
    Trigger a browser download from a Colab notebook cell.
    Usage: download_from_colab(str(OUTPUT_DIR / 'latest_enhanced.png'))
    """
    try:
        from google.colab import files
        files.download(file_path)
        print(f"Downloading: {file_path}")
    except ImportError:
        print(f"Not in Colab — file saved at: {file_path}")


print("✅ Output saving system ready.")
print(f"   Output directory: {OUTPUT_DIR.resolve()}")

---
## SECTION 12 — Launch Application

In [ ]:
# ============================================================
# SECTION 12 — Launch Application
# ============================================================
# This cell launches the Gradio web interface.
# In Google Colab, a public share link will be generated.

print("\n" + "="*60)
print("  🎨 Context-Aware Colorisation — Launching GUI")
print("="*60)
print(f"  DeOldify available : {deoldify_model.available}")
print(f"  Device             : {DEVICE}")
print(f"  Output directory   : {OUTPUT_DIR.resolve()}")
print("="*60 + "\n")

demo.launch(
    debug=False,
    share=True,        # generates a public Gradio link for Colab
    server_port=7860,
    show_error=True,
    quiet=False,
)

In [ ]:
# ============================================================
# OPTIONAL — Manual Download
# ============================================================
# Run this cell after processing an image to download the result.

download_from_colab(str(OUTPUT_DIR / "latest_enhanced.png"))

In [ ]:
# ============================================================
# OPTIONAL — Quick Test (no GUI)
# ============================================================
# Run this cell to test the pipeline on a single image
# directly in the notebook without the Gradio interface.

from IPython.display import display
import requests
from io import BytesIO

# Download a sample cityscape image for testing
TEST_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
)

print("Fetching test image...")
try:
    resp = requests.get(TEST_URL, timeout=10)
    test_pil = Image.open(BytesIO(resp.content)).convert("RGB")
    # Convert to grayscale to simulate a real use case
    test_pil = test_pil.convert("L").convert("RGB")
    print(f"Test image size: {test_pil.size}")
except Exception as e:
    # Create a synthetic gradient test image if URL fails
    print(f"URL failed ({e}), generating synthetic test image...")
    arr = np.zeros((256, 512, 3), dtype=np.uint8)
    # Sky gradient
    arr[:100, :] = np.linspace(50, 180, 100)[:, None, None]
    # Ground
    arr[100:180, :] = 60
    arr[180:, :] = 40
    test_pil = Image.fromarray(arr).convert("L").convert("RGB")

print("Running pipeline...")
results = run_pipeline(test_pil)

print(f"\n{'='*40}")
print(f"Scene type      : {results['scene_type']}")
print(f"Processing time : {results['processing_time']}s")
print(f"Objects detected: {results['object_count']}")
print(f"Seg classes     : {results['num_seg_classes']}")
print(f"{'='*40}")

# Display in notebook
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
pairs = [
    (results["gray_pil"],     "Grayscale Input"),
    (results["seg_vis_pil"],  "Segmentation Map"),
    (results["deoldify_pil"], "DeOldify"),
    (results["enhanced_pil"], "Context-Enhanced"),
]
for ax, (img, title) in zip(axes, pairs):
    ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "quick_test_result.png"), dpi=120, bbox_inches="tight")
plt.show()
print("✅ Quick test complete. Result saved to outputs/quick_test_result.png")